<a href="https://colab.research.google.com/github/evandwh/ST554_Homework/blob/main/Homework_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Assignment: Homework #8

Author: Evan Whitfield

Course: ST554 - Spring 2026

In [223]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, train_test_split, cross_validate
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, log_loss
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV, ElasticNetCV, LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import PolynomialFeatures
import pandas as pd
import numpy as np
from sklearn import linear_model

# HW 7 Starts Here

### Data

We will use a dataset from the UCI Machine Learning Repository. This data set is about wine quality. You can learn more about the data here.

The data description describes the following variables:
**Input variables** (based on physicochemical tests)

1- fixed acidity

2- volatile acidity

3- citric acid

4- residual sugar

5- chlorides

6- free sulfur dioxide

7- total sulfur dioxide

8- density

9- pH

10- sulphates

11- alcohol

**Output variable** (based on sensory data):
12- quality (score between 0 and 10

## Our purpose

Rather than try to predict quality, we will make our target variable for fitting multiple linear regression type models `alcohol`.


For fitting logistic regression type models we’ll use the `type of wine` as the response variable.

## Read In and Combine Data

First, we will read in the CSV files into seperate dataframes. We will also create a new column for the type of wine, so that when we combine the data sets, we will know which type of wine each row is from.

In [306]:
red_wine_data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Colab Data/winequality-red.csv', sep=';')
red_wine_data['type'] = 'red'

white_wine_data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Colab Data/winequality-white.csv', sep=';')
white_wine_data['type'] = 'white'

In [307]:
red_wine_data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red


In [308]:
white_wine_data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6,white
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6,white
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6,white
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6,white
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6,white


Both datasets were read into Python correctly.

Now we will combine the dataframes into one master dataframe called `wine_data`.

In [309]:
wine_data = pd.concat([red_wine_data, white_wine_data])
wine_data["type_num"] = wine_data["type"].map({"red": 0, "white": 1})
wine_data.head(-5)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type,type_num
0,7.4,0.700,0.00,1.90,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red,0
1,7.8,0.880,0.00,2.60,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5,red,0
2,7.8,0.760,0.04,2.30,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5,red,0
3,11.2,0.280,0.56,1.90,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6,red,0
4,7.4,0.700,0.00,1.90,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4888,6.8,0.220,0.36,1.20,0.052,38.0,127.0,0.99330,3.04,0.54,9.2,5,white,1
4889,4.9,0.235,0.27,11.75,0.030,34.0,118.0,0.99540,3.07,0.50,9.4,6,white,1
4890,6.1,0.340,0.29,2.20,0.036,25.0,100.0,0.98938,3.06,0.44,11.8,6,white,1
4891,5.7,0.210,0.32,0.90,0.038,38.0,121.0,0.99074,3.24,0.46,10.6,6,white,1


## Split the Data

We are going to split up the data set into a training and test set. For this, we will use stratified sampling to
make sure that you have a similar proportion of white and red wines in the training and test sets.

In [331]:
X = wine_data.drop(columns=["alcohol","type"])
y = wine_data[["alcohol", "type"]]

X_train_temp, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify = wine_data['type'])

In [332]:
print(X_train.head())

      fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
2511      -0.780764         -0.235477     1.194055        1.228993   0.054215   
1463       0.833280         -0.903574     1.194055       -0.887078  -0.142385   
3637       1.140718         -0.235477     0.150435       -0.125293  -1.069211   
497       -0.012171          0.007468     0.011286       -0.611989   0.952956   
826        0.218406         -0.417685     0.150435       -0.654311  -0.170470   

      free sulfur dioxide  total sulfur dioxide   density        pH  \
2511             2.108548              1.748933  0.623708 -0.174685   
1463            -1.098543              0.374452 -0.953508 -0.986425   
3637            -0.423366              0.198236  0.020655 -0.611776   
497              0.701929             -0.048465  0.636962  0.637055   
826             -1.492396             -1.898728  0.139940  1.136588   

      sulphates   quality  type_num  
2511   0.465326  0.201631  0.571296  
1463  -0.4

In [333]:
print(X_test.head())

      fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
140             6.3              0.31         0.34             2.2      0.045   
1246            8.0              0.20         0.40             5.2      0.055   
3683            6.8              0.15         0.41            12.9      0.044   
4223            5.3              0.43         0.11             1.1      0.029   
3638            6.4              0.29         0.32             2.4      0.014   

      free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
140                  20.0                  77.0  0.99270  3.30       0.43   
1246                 41.0                 167.0  0.99530  3.18       0.40   
3683                 79.5                 183.0  0.99742  3.24       0.78   
4223                  6.0                  51.0  0.99076  3.51       0.48   
3638                 34.0                  89.0  0.99008  3.24       0.66   

      quality  type_num  
140         5         1 

## Regression Models

We will create a set for y_train and y_test for the .  x_train and x_test will change based on the model/variables we choose for each task.

### 1 - MLR Models

In [343]:
x_means = X_train_temp.mean(axis = 0)
x_stds = X_train_temp.std(axis=0)

#Standardizing
X_train = (X_train_temp - x_means)/x_stds

print(x_means)

fixed acidity             7.215836
volatile acidity          0.338770
citric acid               0.318378
residual sugar            5.392101
chlorides                 0.056070
free sulfur dioxide      30.524533
total sulfur dioxide    115.750337
density                   0.994678
pH                        3.217976
sulphates                 0.531155
quality                   5.823167
type_num                  0.753896
dtype: float64


In [261]:
reg_train = y_train["alcohol"]
class_train = y_train["type"]

In [345]:
X_train.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality,type_num
2511,-0.780764,-0.235477,1.194055,1.228993,0.054215,2.108548,1.748933,0.623708,-0.174685,0.465326,0.201631,0.571296
1463,0.833280,-0.903574,1.194055,-0.887078,-0.142385,-1.098543,0.374452,-0.953508,-0.986425,-0.480932,0.201631,0.571296
3637,1.140718,-0.235477,0.150435,-0.125293,-1.069211,-0.423366,0.198236,0.020655,-0.611776,-0.278163,1.341868,0.571296
497,-0.012171,0.007468,0.011286,-0.611989,0.952956,0.701929,-0.048465,0.636962,0.637055,1.749533,-0.938606,-1.750069
826,0.218406,-0.417685,0.150435,-0.654311,-0.170470,-1.492396,-1.898728,0.139940,1.136588,0.735685,1.341868,-1.750069


In [344]:
reg_train.head()

,alcohol
2511,9.4
1463,11.0
3637,11.2
497,11.1
826,11.0


### Linear Full Model

Using a normal linear model without removing any of the variables.

In [262]:
lr_full = LinearRegression()
lr_full.fit(X_train, reg_train)

cv_full_model = cross_validate(
    lr_full,
    X_train,
    reg_train,
    cv=5,
    scoring = "neg_mean_squared_error"
)

### Interaction Terms

We still will not remove any variables, but will now introduce interation terms between the variables.

In [264]:
# Add interaction terms
poly_interation = PolynomialFeatures(
    degree=2,
    interaction_only=True,
    include_bias=False
)
X_interaction = poly_interation.fit_transform(X_train)

inter_model = LinearRegression()
inter_model.fit(X_interaction, reg_train)

# Cross-validate
cv_interaction_model = cross_validate(
    LinearRegression(),
    X_interaction,
    reg_train,
    cv=5,
    scoring="neg_mean_squared_error"
)

### Polynomial Terms

Just because I want to see what happens, I am going to (most likely) over train our model with degree 2 terms with interaction_only = False. I want the model to square some terms to see what happens. Most likely, some of the variables need to be removed, but lets give it a shot first!

In [265]:
# Add interaction terms
poly_ = PolynomialFeatures(
    degree=2,
    interaction_only=False,
    include_bias=False
)
X_poly = poly_.fit_transform(X_train)

poly_model = LinearRegression()
poly_model.fit(X_poly, reg_train)

# Cross-validate
cv_poly_model = cross_validate(
    poly_model,
    X_poly,
    reg_train,
    cv=5,
    scoring="neg_mean_squared_error"
)

### Best Guess

After doing some exploring on the internet, it seems like density and sugar are the most important factors in determining alcohol level, so we will use those only in our last prediction model.

In [266]:
X_best_maybe = poly_interation.fit_transform(X_train[["residual sugar", "density"]])

best_maybe_model = LinearRegression()
best_maybe_model.fit(X_best_maybe, reg_train)

cv_best_maybe_model = cross_validate(
    best_maybe_model,
    X_best_maybe,
    reg_train,
    cv=5,
    scoring="neg_mean_squared_error"
)

In [267]:
print(np.sqrt(-sum(cv_full_model['test_score'])),
      np.sqrt(-sum(cv_interaction_model['test_score'])),
      np.sqrt(-sum(cv_poly_model['test_score'])),
      np.sqrt(-sum(cv_best_maybe_model['test_score'])))

1.1548944121374503 1.0429122087484146 0.9348946913871552 2.108284217349092


It seems like our `X_best_maybe` was in fact not the best, but it might be the best for only two variables in the data. Obviously, the other models also include the `residual sugar` and `density` variables in them as well, so the extra variables seem to cause the mean_squared_error to be lower for predicting `alcohol`.

If given enough time, I will make a script that will test any combination of 2 variables in our data set to see which two variables give the lowest RMSE.

The polynomial model that included interaction and none-interaction terms of degree two had the lowest RMSE at 0.935.

In [272]:
mlr_best = poly_model

vars3 = ['fixed acidity', 'pH','residual sugar', 'chlorides', 'density']

mlr_best.fit(X_train[vars3], y_train['alcohol'])

LinearRegression()

We determined a list of five variables to move forward with, picked the best performing model, and fit the model to our training data.

### Bonus just to test some things out

In [ ]:
#####################
#
# Bonus Time
#
#######################
from iter import combinations

original_vars = wine_data.columns.tolist()
original_vars.remove("alcohol")
original_vars.remove("type")

linear_list = []

for r in range(1, 3):
    for combo in combinations(original_vars, r):
        cv_model = cross_validate(
            LinearRegression(),
            X_train_reg[list(combo)],
            y_train,
            cv=5,
            scoring="neg_mean_squared_error"
        )
        score = np.sqrt(-sum(cv_model['test_score']))
        linear_list.append((combo, score))

sorted_list = sorted(linear_list, key=lambda x: x[1])

print(sorted_list[0])

(('density', 'type_num'), np.float64(1.8212508331148969))


It seems like Density and Type produce the smallest RMSE based on solely the two variables and no interaction terms! Type was just 0 for red and 1 for white. This reveals that the type of wine has a big impact on predicting the amount of alcohol!

In [ ]:
inter_list = []

for r in range(1, 3):
    for combo in combinations(original_vars, r):
        X_train_interations = poly_interation.fit_transform(X_train_reg[list(combo)])
        cv_model = cross_validate(
            LinearRegression(),
            X_train_interations,
            y_train,
            cv=5,
            scoring="neg_mean_squared_error"
        )
        score = np.sqrt(-sum(cv_model['test_score']))
        inter_list.append((combo, score))

sorted_inter_list = sorted(inter_list, key=lambda x: x[1])

print(sorted_inter_list[0])

(('fixed acidity', 'density'), np.float64(1.7957911073192032))


It seems like Fixed Acidity and Density produced the lowest RMSE when interaction terms were introduced. I have no clue at what the easiest things to measure are for wine. So is this better? I do not know, but I do know it has a slightly lower RMSE than just Density and Quality without an interaction term! Type of Wine is probably the easiest thing to "measure" for the wine in our prediction.

## LASSO-ing a Model

Now we are going to attempt to do the problem using a LASSO model.

In [274]:
lasso_model = LassoCV(cv=5, random_state = 0) \
    .fit(X_train[vars3],
         reg_train)

In [275]:
lasso_model.alpha_

np.float64(0.0008255721135897031)

In [276]:
lasso_best = linear_model.Lasso(lasso_model.alpha_)
lasso_best.fit(X_train[vars3],
         reg_train)

print(lasso_best.intercept_, lasso_best.coef_)

10.489583092809912 [ 0.84138557  0.57401867  0.7555286   0.11179703 -1.67993342]


I used the variables listed in vars3 in my Lasso Model to try to find the best fit for the data in predicting alcohol content.

Lasso Model appears to be optimized at alpha = 0.0008256. This is slightly larger than 0, but that does produce a different intercept and coefficients than if alpha did equal 0.

### Ridge Regression

In [277]:
ridge_model_1 = RidgeCV(cv=5) \
    .fit(X_train[vars3],
         reg_train)

In [278]:
ridge_model_1.alpha_

np.float64(1.0)

In [280]:
ridge_best_1 = linear_model.Ridge(ridge_model_1.alpha_)
ridge_best_1.fit(X_train[vars3], reg_train)

print(ridge_best_1.intercept_, ridge_best_1.coef_)

10.489583092809912 [ 0.84660688  0.57789782  0.76251012  0.11463007 -1.68778382]


The Ridge Regression was performed much in the same way that the Lasso Method was performed. The alpha came out to be 1.0. However, when I tried to run the RidgeCV command with `alphas =`a different value was returned.

 I'm not sure at this point if it has to do with the variables that I have chosen to produce these results, or if I am doing something incorrect.

In [281]:
ridge_model_2 = RidgeCV(cv=5, alphas = np.linspace(0, 5, 2000)) \
    .fit(X_train[vars3],
         reg_train)

print(ridge_model_2.alpha_)

3.839419709854927


When running the Ridge Model with alphas = np.linspare(0,5,2000), the best alpha was 3.8394. Other combinations returned a different alpha. I'm not entirely sure what is happening in this model for different results to happen, or if it is because of the way that alphas/linspace are working in combination with each other.

In [346]:
ridge_best_2 = linear_model.Ridge(ridge_model_2.alpha_)
ridge_best_2.fit(X_train[vars3],
         reg_train)

print(ridge_best_2.intercept_, ridge_best_2.coef_)

10.489583092809912 [ 0.84227066  0.57501147  0.7569567   0.11284893 -1.681189  ]


When looking at the model intercept and coefficient, you can see that the intercepts are the same for both Ridge Models, but the 'slopes' are slightly different, which should lead to different predictions for each model.

### Elastic Net

Time for the final model we will be testing.

In [288]:
el_net_model = ElasticNetCV(cv=5, random_state=42) \
    .fit(X_train[vars3], reg_train)

# Print the chosen alpha and l1_ratio
print("Alpha:", el_net_model.alpha_)
print("Number of Iterations:", el_net_model.n_iter_)
print("L1 Ratio:", {el_net_model.l1_ratio_})

Alpha: 0.0016511442271794062
Number of Iterations: 43
L1 Ratio: {np.float64(0.5)}


The chosen alpha is 0.00165, with a L1 Ratio if 0.5. It took the program 43 iterations to choose this alpha. The L1 Ratio is the ration between the L1 (Lasso) and L2 (Ridge) penalties.

In [287]:
print(el_net_model.intercept_, el_net_model.coef_)

10.489583092809912 [ 0.8347639   0.56960994  0.74709741  0.10911323 -1.66980411]


Again, all of the slopes look similar to the slopes of the other models with some slight differences.

I chose to do the same variables for each model to make sure that the code was producing similar models.

## Testing Our Models

It is now time to test our models. We will compare our chosen models based on RMSE and MAE.

In [347]:
# Standardize X_test_reg using the means and stds from the training data
def my_std_fun(x, means, stds):
    return(x-means)/stds

#loop through the columns and use the function on each
for x in X_test.columns:
    X_test[x] = my_std_fun(X_test[x], x_means[x], x_stds[x])

models = [mlr_best, lasso_best, ridge_best_1, ridge_best_2, el_net_model]

for model in models:
    preds = model.predict(X_test[vars3])
    rmse = np.sqrt(mean_squared_error(y_test["alcohol"], preds))
    print(f"Model {model}")
    print(f"RMSE: {rmse}")
    print("")

Model LinearRegression()
RMSE: 0.6296162372030844

Model Lasso(alpha=np.float64(0.0008255721135897031))
RMSE: 0.6300393081588773

Model Ridge(alpha=np.float64(1.0))
RMSE: 0.6297021028115485

Model Ridge(alpha=np.float64(3.839419709854927))
RMSE: 0.6299520954300539

Model ElasticNetCV(cv=5, random_state=42)
RMSE: 0.6304608449777661



Based on the RMSE, all of the models were very similar. The MLR with polynomial terms is performing just a tiny little bit better than the rest.

In [348]:
for model in models:
    preds = model.predict(X_test[vars3])
    mae = np.sqrt(mean_absolute_error(y_test["alcohol"], preds))
    print(f"Model {model}")
    print(f"RMSE: {mae}")
    print("")

Model LinearRegression()
RMSE: 0.6903569810421148

Model Lasso(alpha=np.float64(0.0008255721135897031))
RMSE: 0.690650833057779

Model Ridge(alpha=np.float64(1.0))
RMSE: 0.6904283728688085

Model Ridge(alpha=np.float64(3.839419709854927))
RMSE: 0.6906297416184185

Model ElasticNetCV(cv=5, random_state=42)
RMSE: 0.690975852175524



Again, the polynomial model had a slightly lower MAE compared to the other models. All of them were VERY similar yet again.

## Classification Task

Now, we are going to see if we can create an algorithm to predict the type of wine based on some variables about the wine.

In [367]:
log_model_full = LogisticRegression(max_iter = 1000)
log_model_1 = LogisticRegression(max_iter = 1000)
log_model_2 = LogisticRegression(max_iter = 1000)
log_model_3 = LogisticRegression(max_iter = 1000)
log_model_4 = LogisticRegression(max_iter = 1000)

cv_log_full= cross_validate(
    log_model_full,
    X_train_class,
    y_train_class,
    cv=5,
    scoring="neg_log_loss"
)

cv_log_1= cross_validate(
    log_model_1,
    X_train_class[["density", "pH"]],
    y_train_class,
    cv=5,
    scoring="neg_log_loss"
)

cv_log_2= cross_validate(
    log_model_2,
    X_train_class[["density", "residual sugar"]],
    y_train_class,
    cv=5,
    scoring="neg_log_loss"
)

cv_log_3= cross_validate(
    log_model_3,
    X_train_class[["density", "residual sugar", "pH"]],
    y_train_class,
    cv=5,
    scoring="neg_log_loss"
)

vars2 = ['density', 'fixed acidity', 'residual sugar', 'pH']

cv_log_4= cross_validate(
    log_model_4,
    X_train_class[vars2],
    y_train_class,
    cv=5,
    scoring="neg_log_loss"
)

log_lasso = LogisticRegression(
    penalty='l1',
    solver='saga',
    max_iter=1000
)

cv_log_lasso = cross_validate(
    log_lasso,
    X_train[vars2],
    class_train,
    cv=5,
    scoring="neg_log_loss"
)

log_ridge = LogisticRegression(
    penalty='l2',
    solver='saga',
    max_iter=1000
)

cv_log_ridge = cross_validate(
    log_ridge,
    X_train[vars2],
    class_train,
    cv = 5,
    scoring = "neg_log_loss"
)

log_el_net = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    max_iter=1000,
    l1_ratio=0.5
)

cv_el_net = cross_validate(
    log_el_net,
    X_train[vars2],
    class_train,
    cv = 5,
    scoring = "neg_log_loss"
)

print("Mean CV Log Loss:", -cv_log_full["test_score"].mean())
print("Mean CV Log Loss:", -cv_log_1["test_score"].mean())
print("Mean CV Log Loss:", -cv_log_2["test_score"].mean())
print("Mean CV Log Loss:", -cv_log_3["test_score"].mean())
print("Mean CV Log Loss:", -cv_log_4["test_score"].mean())
print("Mean CV Log Loss:", -cv_log_lasso["test_score"].mean())
print("Mean CV Log Loss:", -cv_log_ridge["test_score"].mean())
print("Mean CV Log Loss:", -cv_el_net["test_score"].mean())



Mean CV Log Loss: 0.0640445574558753
Mean CV Log Loss: 0.5024110033225874
Mean CV Log Loss: 0.46905917489231336
Mean CV Log Loss: 0.43806138734763067
Mean CV Log Loss: 0.25181270318210924
Mean CV Log Loss: 0.14449713659739122
Mean CV Log Loss: 0.14454241219243752
Mean CV Log Loss: 0.1444784414312406


Not suprisingly, the more variables we include in the predictions, the lower the negative log loss is. However, the Ridge, Lasso, and Elastic Net all produced a very similar log-loss for this dataset and variables.

### Testing

In [ ]:
y_pred_full = log_full.predict(X_test_class)
y_pred_1 = log_1.predict(X_test_class[["density", "pH"]])
y_pred_2 = log_2.predict(X_test_class[["density", "residual sugar"]])
y_pred_3 = log_3.predict(X_test_class[["density", "residual sugar", "pH"]])
y_pred_4 = log_4.predict(X_test_class[['density', 'fixed acidity', 'residual sugar', 'pH']])

In [ ]:
print("Full Model Log Loss:", log_loss(y_test_class, y_pred_full))
print("Full Model Accuracy:", accuracy_score(y_test_class, y_pred_full))

Full Model Log Loss: 0.5822436316703542
Full Model Accuracy: 0.9838461538461538


In [ ]:
print("Model 1 Log Loss:", log_loss(y_test_class, y_pred_1))
print("Model 1 Accuracy:", accuracy_score(y_test_class, y_pred_1))

Model 1 Log Loss: 8.900009798389698
Model 1 Accuracy: 0.7530769230769231


In [ ]:
print("Model 2 Log Loss:", log_loss(y_test_class, y_pred_2))
print("Model 2 Accuracy:", accuracy_score(y_test_class, y_pred_2))

Model 2 Log Loss: 8.8722839111673
Model 2 Accuracy: 0.7538461538461538


In [ ]:
print("Model 3 Log Loss:", log_loss(y_test_class, y_pred_3))
print("Model 3 Accuracy:", accuracy_score(y_test_class, y_pred_3))

Model 3 Log Loss: 8.650476813388117
Model 3 Accuracy: 0.76


In [ ]:
print("Model 4 Log Loss:", log_loss(y_test_class, y_pred_4))
print("Model 4 Accuracy:", accuracy_score(y_test_class, y_pred_4))

Model 4 Log Loss: 4.2420607450268655
Model 4 Accuracy: 0.8823076923076923


The full model had by far the lowest log loss, and the best accuracy score. My 2nd, 3rd, and 4th model were all pretty similar to one another.

The accuracy and log loss seem to improve with more variables included. I have no idea how easy/hard to get thses measurements. If I knew how easy it would be to record various vales for your wine, then I could look and compare the log loss and accuracy of only using easy to measure variables.

# HW8 Starts



## Regression Tree (RMSE)

First, I am going to pick 5 variables to use as our predictors.

Then, set the parameters for the regression/classification trees.

In [232]:
vars3 = ['fixed acidity', 'pH','residual sugar', 'chlorides', 'density']

parameters = {'max_depth': range(2,15),
              'min_samples_leaf':[3, 5, 10, 50, 100]}

rtree_tune = GridSearchCV(DecisionTreeRegressor(),
                            parameters,
                            cv = 5,
                            scoring= 'neg_mean_squared_error') \
                            .fit(X_train[vars3], y_train)

print(rtree_tune.best_estimator_)

DecisionTreeRegressor(max_depth=13, min_samples_leaf=5)


The best estimator values are a max depth of 13 and a min samples per leaf of 5.

## Random Forest (RMSE)

In [ ]:
parameters_2 = {"max_features": range(1,3), "max_depth": [3, 4, 5, 10],'min_samples_leaf':[3, 10, 50]}

rfor_tune = GridSearchCV(RandomForestRegressor(n_estimators = 100),
                         parameters_2,
                         cv = 5,
                         scoring = "neg_mean_squared_error",
                         n_jobs = -1) \
                         .fit(X_train[vars3], y_train)

In [ ]:
print(rfor_tune.best_estimator_)

RandomForestRegressor(max_depth=10, max_features=2, min_samples_leaf=3)


The best estimator values are: max depth is 10, max features is 2, and min samples per leaf is 3.

## Testing Models to Compare RMSE and MAE

In [351]:
models = {
    "Regression Tree" : rtree_tune,
    "Randomized Forest" : rfor_tune,
    "Multiple Linear Regression" : mlr_best,
    "Lasso" : lasso_best,
    "Ridge #1" : ridge_best_1,
    "Ridge #2" : ridge_best_2,
    "Elastic Net" : el_net_model
}

for name, model in models.items():
    preds = model.predict(X_test[vars3])
    rmse = round(np.sqrt(mean_squared_error(y_test['alcohol'], preds)),4)
    mae = round(np.sqrt(mean_absolute_error(y_test['alcohol'], preds)),4)
    print(f"Model: {name}")
    print(f"RMSE: {rmse}")
    print(f"MAE: {mae}")
    print("")

Model: Regression Tree
RMSE: 2.9112
MAE: 1.5986

Model: Randomized Forest
RMSE: 2.0231
MAE: 1.3099

Model: Multiple Linear Regression
RMSE: 0.6296
MAE: 0.6904

Model: Lasso
RMSE: 0.63
MAE: 0.6907

Model: Ridge #1
RMSE: 0.6297
MAE: 0.6904

Model: Ridge #2
RMSE: 0.63
MAE: 0.6906

Model: Elastic Net
RMSE: 0.6305
MAE: 0.691



As expected, the randomized forest performed better than the Regression Tree in RMSE and MAE. However, there was not a huge difference between the two models in the new addition of the homework.

However, the models in HW7 faired much better. The MLR with the polynomial terms still has the lowest RMSE and MAE.

## Classification Task


Repeating the training and testing done previously but use classification trees and random forests with classification trees.

We will use negative log-loss as our metric for choosing models during the training process.

In [ ]:
rtree_nll = GridSearchCV(DecisionTreeClassifier(),
                         parameters,
                         cv = 5,
                         scoring= 'neg_log_loss') \
                         .fit(X_train_class[vars3], y_train_class)

print(rtree_nll.best_estimator_)

rfor_nll = GridSearchCV(RandomForestClassifier(),
                        parameters_2,
                        cv = 5,
                        scoring= 'neg_log_loss') \
                        .fit(X_train_class[vars3], y_train_class)

print(rfor_nll.best_estimator_)

DecisionTreeClassifier(max_depth=5, min_samples_leaf=50)
RandomForestClassifier(max_depth=10, max_features=2, min_samples_leaf=3)


### Testing
Comparing both of our models on both log-loss and accuracy.

In [246]:
models = {
    "Regression Tree": rtree_nll,
    "Random Forest": rfor_nll
}

for name, model in models.items():
    preds = model.predict(X_test_class[vars3])
    print(f"Model: {name}")
    print(f"Log Loss =", log_loss(y_test_class, preds))
    print(f"Accurary =", accuracy_score(y_test_class,preds))
    print("")

Model: Regression Tree
Log Loss = 1.608101458899073
Accurary = 0.9553846153846154

Model: Random Forest
Log Loss = 0.5267918572255587
Accurary = 0.9853846153846154



As expected, the random forest produced a lower log loss and a higher accuracy than the singular classification tree.

These log-loss scores do not seem to be as log as the log loss produced by the models in HW7. The SLR with all of the variables still performs the best. This should be expected to some extent. It seems like using all the variables does not overtrain the data.